# SAM Predictions vs Ground Truth Annotations

This notebook evaluates Segment Anything Model (SAM) predictions against COCO ground truth masks.

## Evaluation Metrics:
- **IoU (Intersection over Union)**: Overlap between predicted and ground truth masks
- **Dice Coefficient**: Similarity measure for segmentation
- **Precision & Recall**: Per-pixel classification accuracy
- **Boundary F1 Score**: Edge quality metric
- **Mean metrics across dataset samples**

## 1. Install Dependencies

In [9]:
# from code.MaskRefiner import MaskRefiner
!pip install segment-anything git+https://github.com/facebookresearch/segment-anything.git
!pip install opencv-python matplotlib pycocotools scikit-image pandas seaborn

  Cloning https://github.com/facebookresearch/segment-anything.git to /private/var/folders/gx/5qw2j36n0d14_dkv7wr87tfw0000gn/T/pip-req-build-rypxe3t9
  Running command git clone --filter=blob:none --quiet https://github.com/facebookresearch/segment-anything.git /private/var/folders/gx/5qw2j36n0d14_dkv7wr87tfw0000gn/T/pip-req-build-rypxe3t9
  Resolved https://github.com/facebookresearch/segment-anything.git to commit dca509fe793f601edb92606367a655c15ac00fdf
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


## 2. Import Libraries

In [10]:
import torch
import numpy as np
from segment_anything import sam_model_registry, SamPredictor
import urllib.request
import os
import pandas as pd
from tqdm import tqdm
import json
from pycocotools.coco import maskUtils
from coding.FunctionUtils import load_sa1b_image_and_annotations
# from google.colab import drive
# drive.mount('/content/drive')

In [11]:
image_dir = '/Users/carricarte/PhD/Projects/MARL-med/scratch/dataset'
output_dir = '/Users/carricarte/PhD/Projects/MARL-med/scratch/dataset'
file_path = "/Users/carricarte/PhD/Projects/MARL-med/scratch"
# output_dir = '/content/drive/MyDrive/marl/'
threshold = 0.815 # Threshold for challenging segmentations
files = []
[files.append(os.path.join(file_path, f)) for f in os.listdir(file_path) if f.endswith(".csv") and "._" not in f]

[None]

## 3. Download SAM Model

In [12]:
# !nvidia-smi
# Choose model size: 'vit_h' (best), 'vit_l', or 'vit_b' (fastest)
model_type = "vit_b"

checkpoint_url = {
    'vit_h': 'https://dl.fbaipublicfiles.com/segment_anything/sam_vit_h_4b8939.pth',
    'vit_l': 'https://dl.fbaipublicfiles.com/segment_anything/sam_vit_l_0b3195.pth',
    'vit_b': 'https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth'
}

checkpoint_path = f"sam_{model_type}.pth"

if not os.path.exists(checkpoint_path):
    print(f"Downloading {model_type} checkpoint...")
    urllib.request.urlretrieve(checkpoint_url[model_type], checkpoint_path)
    print("Download complete!")

# Initialize SAM
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

sam = sam_model_registry[model_type](checkpoint=checkpoint_path)
sam.to(device=device)
predictor = SamPredictor(sam)

print("SAM model loaded!")

Using device: cpu
SAM model loaded!


In [13]:
# activations = {}
#
# def get_activation(name):
#     def hook(model, input, output):
#         activations[name] = output.detach()  # ← Actually calls the method
#     return hook  # ← Returns the hook function
#
#
# for i, block in enumerate(sam.image_encoder.blocks):
#     block.register_forward_hook(get_activation(f'block_{i}'))

In [19]:
def save_masks_and_embeddings(all_files):
    box_batch_size = 32
    file_no = 1

    for my_file in tqdm(all_files, desc="Processing CSV files"):
        try:
            df = pd.read_csv(my_file)
            df_challenging_mask = df[df['IoU'] <= threshold]

            all_pred_masks = []
            all_embeddings = {}

            unique_images = df_challenging_mask['image_file'].unique()
            json_data = []

            for idx, image_filename in enumerate(tqdm(unique_images, desc="Processing images", leave=False)):
                filename = os.path.splitext(image_filename)[0]
                json_file = os.path.join(image_dir, filename + '.json')

                if not os.path.exists(json_file):
                    continue

                try:
                    # Load image and annotations
                    image, data = load_sa1b_image_and_annotations(json_file)
                    predictor.set_image(image)

                    # Get annotations for this specific image
                    annotations = pd.DataFrame(data['annotations'])
                    challenging_ann_ids = df_challenging_mask[
                        df_challenging_mask['image_file'] == image_filename
                    ]['annotation'].values  # Use .values instead of column

                    annotations = annotations[annotations['id'].isin(challenging_ann_ids)]

                    if len(annotations) == 0:
                        continue

                    # Vectorized bbox creation (faster than iterrows)
                    bboxes = annotations[['bbox']].values
                    boxes = np.array([[b[0][0], b[0][1],
                                      b[0][0] + b[0][2],
                                      b[0][1] + b[0][3]]
                                     for b in bboxes])

                    annotation_ids = annotations['id'].tolist()  # Pre-convert to list
                    gt_segmentations = annotations['segmentation'].tolist()

                    # Process boxes in batches
                    with torch.no_grad():
                        for batch_start in range(0, len(boxes), box_batch_size):
                            batch_end = min(batch_start + box_batch_size, len(boxes))
                            batch_boxes = boxes[batch_start:batch_end]

                            for local_idx, box in enumerate(batch_boxes):
                                global_idx = batch_start + local_idx
                                masks, _, _ = predictor.predict(
                                    box=box[None, :],
                                    multimask_output=False
                                )

                                pred_rle = maskUtils.encode(np.asfortranarray(masks[0].astype(np.uint8)))

                                json_item = {
                                        "image": filename,
                                        "id": annotation_ids[global_idx],
                                        "p_segmentation": {
                                            "size": pred_rle["size"],
                                            "counts": pred_rle["counts"].decode('utf-8')
                                        },
                                        "gt_segmentation": gt_segmentations[global_idx]
                                    }
                                json_data.append(json_item)
                                # all_pred_masks.append({
                                #     "image": filename,
                                #     "id": annotation_ids[global_idx],
                                #     "p_segmentation": maskUtils.encode(np.asfortranarray(masks[0].astype(np.uint8))),
                                #     "gt_segmentation": gt_segmentations[global_idx]
                                # })

                    # Save embedding once per image (not per annotation)
                    # block_name = "block_10"
                    # if block_name in activations:
                    #     block_embedding = activations[block_name]
                    #     flattened_embedding = block_embedding.flatten().detach().cpu().numpy()
                    #     all_embeddings[filename] = flattened_embedding

                except Exception as e:
                    print(f"Error processing {filename}: {e}")
                    continue

                # Save checkpoint every 100 images
                if (idx + 1) % 100 == 0:

                    with open(f'{output_dir}/challenging_masks_{file_no:06d}.json', 'w') as f:
                        json.dump(json_data, f)

                    json_data = []

                    # Save embeddings in batch
                    if all_embeddings:
                        np.savez_compressed(
                            f'{output_dir}/embeddings_{file_no:06d}.npz',
                            **all_embeddings
                        )

                    print(f"Checkpoint {file_no}: {len(all_pred_masks)} masks, {len(all_embeddings)} embeddings")

                    all_pred_masks = []
                    all_embeddings = {}
                    file_no += 1

            # Save any remaining data at end of file
            if json_data:

                with open(f'{output_dir}/challenging_masks_{file_no:06d}.json', 'w') as f:
                    json.dump(json_data, f)

                if all_embeddings:
                    np.savez_compressed(
                        f'{output_dir}/embeddings_{file_no:06d}.npz',
                        **all_embeddings
                    )

                print(f"Final: {len(all_pred_masks)} masks, {len(all_embeddings)} embeddings")
                file_no += 1

        except Exception as e:
            print(f"Error processing file {my_file}: {e}")
            import traceback
            traceback.print_exc()
            continue

# Run evaluation
print("Getting challenging masks and image embeddings...")
save_masks_and_embeddings(files)

Getting challenging masks and image embeddings...


Processing CSV files:   0%|          | 0/1 [02:09<?, ?it/s]A


KeyboardInterrupt: 